In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import uniform, randint
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve
import joblib, os, numpy as np, pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hgb = HistGradientBoostingClassifier(random_state=42)

In [ ]:
# Load test data
X_train = pd.read_parquet("../data/dataset/X_train.parquet")
y_train = pd.read_parquet("../data/dataset/y_train.parquet").squeeze()
X_test = pd.read_parquet("../data/dataset/X_test.parquet")
y_test = pd.read_parquet("../data/dataset/y_test.parquet").squeeze()

# Load trained model
hgb_model = joblib.load("../data/models/HistGradientBoosting.pkl")

## 1. RandomizedSearch

In [ ]:
param_dist = {
    "learning_rate": uniform(0.01, 0.3),
    "max_iter": randint(100, 1000),
    "max_leaf_nodes": randint(15, 255),
    "min_samples_leaf": randint(1, 50),
    "l2_regularization": uniform(0.0, 1.0)
}

rs = RandomizedSearchCV(
    hgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rs.fit(X_train, y_train)
print("Randomized best params:", rs.best_params_, "best score:", rs.best_score_)

### 2. GridSearch

In [ ]:
best = rs.best_params_
param_grid = {
    "learning_rate": sorted({max(0.001, best["learning_rate"]*f) for f in [0.5, 1.0, 1.5]}),
    "max_iter": sorted({max(50, best["max_iter"] + d) for d in [-100, 0, 100]}),
    "max_leaf_nodes": sorted({max(2, best["max_leaf_nodes"] + d) for d in [-32, 0, 32]})
}

gs = GridSearchCV(hgb, param_grid=param_grid, scoring="average_precision", cv=cv, n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)
print("Grid best params:", gs.best_params_, "best score:", gs.best_score_)

In [ ]:
cal = CalibratedClassifierCV(gs.best_estimator_, method="isotonic", cv=5)
cal.fit(X_train, y_train)

In [ ]:
# 4) Evaluar en X_test y elegir umbral clínico con tu función find_best_threshold
def find_best_threshold(y_true, y_prob, min_recall=0.85):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    best_t, best_score = None, -1
    best_p, best_r = None, None
    # iterate excluding last precision/recall pair (no threshold)
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        f1 = 2 * (p * r) / (p + r + 1e-9)
        if r >= min_recall and f1 > best_score:
            best_score = f1
            best_t = t
            best_p = p
            best_r = r
        elif best_t is None and f1 > best_score:
            # track best overall in case no threshold reaches min_recall
            best_score = f1
            best_p = p
            best_r = r
    return best_t, best_p, best_r, best_score

probs = cal.predict_proba(X_test)[:,1]
t, p, r, f1 = find_best_threshold(y_test, probs, min_recall=0.85)  # ajusta min_recall si hace falta
print("Chosen threshold:", t, "precision:", p, "recall:", r, "f1:", f1)

In [ ]:
# 5) Guardar el modelo calibrado entrenado con todo X_train
output_dir = "../data/models"
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, "HistGradientBoosting_calibrated.pkl")
joblib.dump(cal, model_path)
print("Saved calibrated model to", model_path)